## Analisis Sentimen Aplikasi Kopi Kenangan
### Data Preprocessing (Functional Approach)

In [ ]:
!pip install Sastrawi

In [ ]:
import os
import re
import string
import json
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

### Kamus Slang (Slang Dictionary)

In [ ]:
# Load baseline slang
gmaps_slang_path = os.path.join('..', '..', 'preprocessing', 'combined_slang_words.txt')
with open(gmaps_slang_path, 'r', encoding='utf-8') as f:
    slang_dict = json.load(f)

# Load app-specific slang and merge
app_slang_path = 'slang_tambahan_app.txt'
with open(app_slang_path, 'r', encoding='utf-8') as f:
    app_slang = json.load(f)
slang_dict.update(app_slang)

print(f"Kamus slang didefinisikan dengan {len(slang_dict)} kata.")

### Fungsi Pembersih Teks & Preprocessing

In [ ]:
stopwords_path = os.path.join('..', '..', 'preprocessing', 'combined_stop_words.txt')
with open(stopwords_path, 'r', encoding='utf-8') as f:
    stopword_indo = set([line.strip() for line in f if line.strip()])

print(f"Stopwords didefinisikan dengan {len(stopword_indo)} kata.")

print("Membuat Sastrawi Stemmer...")
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stem_cache = {}

In [ ]:
def get_cached_stem(word, stemmer_obj):
    if not word:
        return ""
    if word not in stem_cache:
        stem_cache[word] = stemmer_obj.stem(word)
    return stem_cache[word]

In [ ]:
def clean_noise(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.replace('\n', ' ')
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    return text.lower()

In [ ]:
def preprocess_review(text):
    try:
        cleaned = clean_noise(text)
        if not cleaned:
            return ""
        tokens = word_tokenize(cleaned)
        normalized = [slang_dict.get(t, t) for t in tokens]
        filtered = [t for t in normalized if t not in stopword_indo]
        stemmed = [get_cached_stem(t, stemmer) for t in filtered if t]
        return ' '.join(stemmed)
    except Exception as e:
        print(f"[ERROR] Failed to process review: '{text[:100]}...' - Error: {e}")
        return ""

### Load Dataset & Jalankan Pipeline

In [ ]:
dataset_path = os.path.join('..', 'scraping', 'raw_app_reviews.csv')
df = pd.read_csv(dataset_path)
print(f"Dataset berhasil dimuat! Jumlah data: {len(df)} baris.")
df.head(2)

In [ ]:
print("Menjalankan text preprocessing pada seluruh ulasan...")
df['text_clean'] = df['review_text'].apply(preprocess_review)
print("Pembersihan selesai!")

In [ ]:
empty_cleaned_reviews = df[df['text_clean'] == ''].shape[0]
total_reviews = df.shape[0]
print(f"Jumlah ulasan yang menjadi kosong setelah preprocessing: {empty_cleaned_reviews} dari {total_reviews}")

### Inspecting Preprocessing Results

In [ ]:
print("Original vs. Preprocessed Reviews (First 10 samples):")
for i in range(10):
    original_review = df['review_text'].iloc[i]
    preprocessed_review = df['text_clean'].iloc[i]
    print(f"\nReview #{i+1}:")
    print(f"  Original: {original_review}")
    print(f"  Cleaned:  {preprocessed_review}")

### Pelabelan Sentimen berdasarkan Rating & Filtering

In [ ]:
df['word_count'] = df['text_clean'].apply(lambda x: len(str(x).split()))

In [ ]:
print(f"Jumlah baris sebelum difilter: {len(df)}")
df_filtered = df[df['word_count'] >= 1].copy()
print(f"Jumlah baris setelah difilter (minimal 1 kata): {len(df_filtered)}")

In [ ]:
def map_rating_to_sentiment(stars):
    try:
        stars = int(stars)
        if stars >= 4:
            return 'positive'
        elif stars <= 2:
            return 'negative'
    except Exception:
        pass
    return 'neutral'

In [ ]:
df_filtered['sentiment'] = df_filtered['rating'].apply(map_rating_to_sentiment)
print("\nDistribusi Kelas Sentimen:")
print(df_filtered['sentiment'].value_counts())

### Hasil Preprocessing

In [ ]:
output_path = 'cleaned_app_reviews.csv'
df_filtered.to_csv(output_path, index=False)
print(f"Proses preprocessing berhasil diselesaikan!")
print(f"Data bersih disimpan ke: {output_path}")